In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
table {margin-left:0 !important; margin-right: auto;}
</style>
"""))

<font size="5" color="black">ch6. OpenAI Whisper API</font>

# OpenAI Whisper API를 활용한 음성-텍스트 변환

OpenAI의 Whisper 모델은 사람의 음성을 높은 정확도로 텍스트로 변환하는 자동 음성 인식(ASR) 모델입니다. Whisper는 OpenAI API를 통해 제공되며, 음성 파일을 원어 그대로 **텍스트로 필사(transcription)** 하거나 영어로 **번역(translation)** 하는 기능을 제공합니다. 이 튜토리얼에서는 Whisper API를 사용하여 음성-텍스트 변환을 구현하는 방법을 단계별로 살펴보겠습니다. 환경 설정부터 기본 사용 방법, 그리고 언어 감지, 타임스탬프 포함 출력, 실시간 음성 처리와 같은 고급 기능까지 다룹니다.

# 1. 환경 설정

* python-dotenv 패키지를 통한 .env 파일 내부 환경 변수 로드
* OPENAI_API_KEY 환경 변수 설정 확인 또는 openai.OpenAI() 클래스 인스턴스 생성 시 api_key 매개변수를 통한 명시적 키 전달
* openai.OpenAI() 클래스 호출을 통한 API 클라이언트 인스턴스 생성

```
환경 변수 또는 인자로 API 키를 지정하지 않으면 OpenAI 라이브러리는 다음과 같은 오류를 발생시킵니다:
```
> OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable


In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

```
위 코드에서는 .env에서 불러온 API 키로 client 객체를 생성했습니다. 이제 이 client를 통해 DALL-E를 비롯한 OpenAI API 요청을 보낼 수 있습니다. (참고로, api_key를 생략하면 OPENAI_API_KEY 환경 변수를 자동으로 참조합니다.)
```

# 2. Whisper API 소개 및 기본 음성 → 텍스트 변환

* OpenAI Whisper API의 두 가지 주요 엔드포인트(Transcriptions, Translations) 제공
    * Transcriptions 엔드포인트 : 음성 파일의 원어 텍스트 필사 기능
    * Translations 엔드포인트 : 음성 파일의 영어 번역 텍스트 반환 기능
* Whisper v2 대형 모델(whisper-1) 기반 처리 및 다양한 음성 파일 형식(MP3, MP4, WAV 등) 지원
* 단일 요청 당 최대 25MB의 파일 크기 제한 및 대용량 오디오 분할 처리 필요성


## 기본 사용법: 음성 파일 필사 요청

* 변환 대상 오디오 파일(WAV, MP3 등) 준비 및 경로 지정
* OpenAI Python SDK(`client.audio.transcriptions.create`)를 통한 API 요청 구성
* `file` ("rb" 모드 파일 객체) 및 `model` ("whisper-1") 등 주요 파라미터 설정
* `response_format` 파라미터를 활용한 응답 형식(JSON, text, srt 등) 변경 지원
* API의 음성-텍스트 변환 결과 반환 및 출력/변수 저장 처리

In [ ]:
# 1. 음성 파일 열기
audio_file = open('data/speech.mp3', 'rb')

# 2. 음성 파일 필사 요청
response = client.audio.transcriptions.create(
    model='whisper-1',
    file=audio_file,
    response_format='text' # 단순 텍스트 형태의 결과 반환 옵션 지정
)

# 3. 변환된 텍스트 출력
print(response)
audio_file.close()

In [ ]:
from IPython.display import Audio
Audio(filename='data/speech.mp3', autoplay=True)

In [ ]:
# with 구문 사용
with open('data/speech.mp3', 'rb') as audio_file:
    response = client.audio.transcriptions.create(
    model='whisper-1',
    file=audio_file,
    response_format='text'
    )
    
print(response)

```
위 코드가 실행되면, speech.mp3 파일의 음성이 텍스트로 변환되어 response에 담기고, 그 내용을 출력합니다. 예를 들어 입력 음성이 영어로 "OpenAI Whisper is an advanced speech recognition model." 라는 문장을 담고 있었다면, 출력은 다음과 같은 텍스트가 될 것입니다:
```
```
OpenAI Whisper is an advanced speech recognition model.
```


# 3. 다양한 언어의 음성 인식 및 언어감지

* Whisper 모델의 다국어 음성 인식 지원
    * 90개 이상의 언어 데이터 훈련을 통한 다양한 언어의 음성-텍스트 변환 기능 제공
    * API 내 언어 미지정 시 Whisper의 음성 언어 자동 감지 및 해당 언어 기반 텍스트 필사 실행
    * 한국어 음성 입력 시 자동 인식을 통한 한국어 텍스트 자동 출력 기능


* 언어 감지 기능 및 `language` 파라미터 활용
* 음성 언어 자동 감지 기능의 한계 및 오인식 가능성 존재
* 오인식 방지를 위한 `language` 파라미터 기반의 입력 음성 언어 직접 지정 기능 제공
* ISO 언어 코드 설정을 통한 특정 언어 인식의 강제화 프로세스
* 한국어 음성 파일 대상 `language="ko"` 지정을 통한 한국어 인식 명시 사례
* 영어(`language="en"`), 스페인어(`language="es"`) 등의 표준 언어 코드를 활용한 언어 설정 사례

In [ ]:
# 스페인어 필사
with open('data/spanish_audio.mp3','rb') as file:
    response = client.audio.transcriptions.create(
        model='whisper-1',
        file=file,
        response_format='text',
        language='es' # 자동 판별 가능하나, 지정 시 정확도 상승
    )

print(response)

```
Whisper에 언어를 지정함으로써 모델이 해당 언어로 정확히 인식하도록 도울 수 있습니다. 대부분의 경우 언어를 지정하지 않아도 잘 동작하지만, 언어 감지 결과를 신뢰하기 어렵거나 특정 언어로 출력되길 원한다면 language 파라미터를 활용하세요.
```

In [ ]:
# 스페인어 해석
with open('data/spanish_audio.mp3','rb') as file:
    response = client.audio.translations.create(
        model='whisper-1',
        file=file,
        response_format='text'
    )

print(response)

In [ ]:
# OpenAI의 whisper는 타국어 → 영어 번역만 가능함

# 타국어 음성 → 한국어 해석 시 해석 - 번역 단계 필요

with open('data/spanish_audio.mp3','rb') as file:
    transcription = client.audio.transcriptions.create(
        model = 'whisper-1',
        file = file,
        response_format='text',
        language='es'
    )
    
print('오디오 해석 진행 중')
translation = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=[{'role':'user','content':f'이 문장을 한국어로 번역해줘 : {transcription}'}]
)

print(translation.choices[0].message.content)

# 4. 타임스탬프 포함 자막 및 세부 출력

* 기본 Whisper API 응답 방식 및 특정 요구 사항
    * 전체 음성을 단일 텍스트 형태로 반환하는 기본 Whisper API 응답 구조
    * 동영상 자막 생성 및 정교한 음성 분석 등의 목적을 위한 문장별 시간 구간 표시 필요성
    * 단어 단위 타임스탬프 정보 추출 요구 사항의 존재

* Whisper API의 다변화된 출력 형식 지원 : 다양한 사용자 및 서비스 요구 조건에 부합하는 여러 가지 출력 형식 제공 기능

## 다양한 출력 형식 (response_format 옵션)

* `response_format` 파라미터를 통한 Whisper 출력 형식 변경
    * 파라미터 조정을 통한 다향한 응답 포맷 설정 기능


* `response_format` 주요 설정 옵션별 특징
    * `"json"` (기본값) : `Transcription(text='OpenAI Whisper is an advanced speech recognition model.',..)` 형식의 기본 객체 반환
    * `"text"` : 순수 변환 텍스트 결과만 단순 반환하는 포맷 (선행 예제 사용)
    * `"srt"` : 문장별 시간 코드가 포함된 SubRip 표준 자막 형식 반환
    * `"vtt"` : SubRip(`srt`)과 유사한 WebVTT 웹 표준 자막 형식 반환
    * `"verbose_json"` : 인식된 문장별 세그먼트, 타임스탬프, 확률 등 상세 메타데이터를 포함한 JSON 반환


* 자막 형식 출력 메커니즘
    * `response_format="srt"` 요청 시 음성 파일 텍스트의 일정 길이 분할 처리
    * 각 분할 구간별 시작 및 종료 시간 정보가 포함된 자막 형식 반환 동작

In [ ]:
with open('data/speech.mp3', 'rb') as file:
    transcript = client.audio.transcriptions.create(
        model='whisper-1',
        file=file
    )
print(transcript)

In [ ]:
Audio('data/speech.mp3', autoplay=True)

In [ ]:
with open('data/speech.mp3', 'rb') as file:
    transcript = client.audio.transcriptions.create(
        model='whisper-1',
        file=file,
        response_format='srt'
    )
print(transcript)

```
응답은 문자열 형태로 SRT 콘텐츠가 반환됩니다.

위 예시처럼, SRT 포맷에서는 각 블록마다 순번, 시작/끝 시간, 그리고 해당 구간의 대사가 나타납니다. Whisper API가 자동으로 음성을 몇 초 단위로 구분해 자막으로 만들어준 것을 확인할 수 있습니다.
```

> 세부 JSON 출력 : 더 구조적인 정보가 필요하다면 response_format="verbose_json"을 사용할 수 있습니다. 이 모드에서는 음성 전체를 몇 개의 **세그먼트(segment)** 로 나누고, 각 세그먼트에 대한 텍스트, 시작/종료 시간 정보(timestamp), 신뢰도 등의 정보를 JSON 형식으로 제공합니다. 이를 활용하면 필요에 따라 세그먼트를 직접 처리하거나 자막을 커스터마이징할 수 있습니다.


In [ ]:
with open('data/ch5_Shimmer_ko_hd.mp3', 'rb') as file:
    transcript = client.audio.transcriptions.create(
        model='whisper-1',
        file=file,
        response_format='verbose_json',
        language='ko'
    )
print(transcript)

In [ ]:
segment = transcript.segments[0]
print(f'text 구간 {segment.start} ~ {segment.end} : {segment}')

```
위 코드에서 transcription_json은 segments에 여러 세그먼트가 리스트로 포함됩니다. 각 세그먼트에는 "start", "end" (초 단위 시간), "text" 등이 담겨 있으며, 이 외에도 "confidence"(평균 예측 확률)나 "language"(감지된 언어) 등의 부가 정보가 있을 수 있습니다.
```

# 5. 실시간 음성 변환 (마이크 입력 처리)

* 실시간 마이크 입력 처리 기술 전환
    * 기존 저장 파일 기반 음성 변환 방식에서 실시간 마이크 입력 변환 방식으로의 확장
    * 실시간 마이크 입력을 수신하여 Whisper API로 직접 전송하는 처리 구조


* 실시간 음성-텍스트 변환(STT) : 발화와 동시에 텍스트로 즉시 변환되는 실시간 처리 프로세스

* Whisper API 기반의 유사 실시간 처리 구현 기법
    * Whisper API 내 자체 스트리밍 엔드포인트 미지원의 한계
    * 짧은 구간 단위 녹음 및 연속적인 API 전송 방식을 통한 유사 실시간 변환 구현

>주의: 네트워크 API를 이용하는 이상 완벽한 실시간(동시에 변환) 처리에는 한계가 있습니다. 음성을 일정 간격으로 끊어서 전송하면 약간의 지연은 발생하지만, 빠른 응답 속도를 감안하면 거의 실시간에 가까운 자막을 구현할 수 있습니다. 진정한 실시간 처리가 필요하다면 오픈소스 Whisper 모델을 로컬에서 사용하거나 스트리밍 기능을 지원하는 서비스를 검토해야 합니다.

## 마이크 입력 녹음하여 전송하기
- sounddevice나 pyaudio 등의 라이브러리

* `sounddevice.rec` 함수를 활용한 짧은 음성 녹음

* `scipy.io.wavfile.write` 함수를 이용한 오디오 데이터의 파일 저장
    * Whisper API 전송을 위한 오디오 데이터의 파일 형태(또는 파일 객체) 제공 필요

* `scipy.io.wavfile.write` 함수를 활용한 녹음 데이터의 WAV 파일 저장 프로세스

* Whisper API를 통한 저장 파일 전송 및 텍스트 변환




In [ ]:
import sounddevice as sd
from scipy.io.wavfile import write

In [ ]:
sf = 16000 # 샘플레이트 16kHz
second = 5 # 음성 녹음길이
print('지금부터 5초간 녹음 시작')

recording = sd.rec(second*sf, samplerate=sf, channels=1, dtype='int16')
sd.wait() # 녹음 종료까지 코드 일시정지

file_name = 'data/ch6_live_input.wav'

write(file_name, sf, recording) # 파일 저장

# Audio(file_name, autoplay=True)

# 저장한 녹음본을 whisper API에 전송하여 text 출력
with open('data/ch6_live_input.wav', 'rb') as audio_file:
    live_transcript = client.audio.transcriptions.create(
        model='whisper-1',
        file=audio_file,
        response_format='text',
        language='ko'
    )

print('실시간 음성 텍스트 :', live_transcript)

# wav 파일 삭제
import os
try:
    os.remove('data/ch6_live_input.wav')
    print('텍스트 생성 후 wav 파일 삭제 완료')
except Exception as e:
    print('이미 삭제된 파일입니다.', e)
    
# 번역
print('오디오 해석 진행 중')
translation = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=[{'role':'user','content':f'이 문장을 영어로 번역해줘 : {live_transcript}'}]
)

print(translation.choices[0].message.content)

# Quiz

1. 과제 개요<br>
교육생들은 OpenAI의 최신 API를 활용하여 주어진 텍스트 파일을 읽고, 내용을 요약한 후, 이를 음성 파일(MP3)로 변환하는 파이썬 프로그램을 개발해야 합니다. <br>
본 과제를 통해 학생들은 다음과 같은 핵심 기술을 학습할 수 있습니다.

- OpenAI API를 활용한 텍스트 요약
- OpenAI의 텍스트 음성 변환(TTS) 기능 적용
- Python 환경에서 파일 입출력 처리
- python-dotenv를 활용한 API 키 관리

2. 요구 사항
``` 
a. 환경변수 사용 : API 키는 코드에 직접 포함하지 않고, .env 파일을 이용해 관리해야 합니다.
b. 텍스트 파일 읽기 : 프로그램은 지정된 텍스트 파일(ch06_quiz.txt)을 읽어와야 합니다.
c. 텍스트 요약 : OpenAI API를 사용하여 긴 텍스트를 100자 이내로 요약해야 합니다.
d. 음성 변환 : 요약된 내용을 OpenAI의 TTS(Text-to-Speech) API를 활용하여 음성데이터로 변환합니다.
e. 지정된 보이스 사용 : OpenAI TTS에서 제공하는 특정 보이스("nova")를 사용해야 합니다.
f. MP3 파일 저장 : 생성된 음성 데이터를 ch6_quiz.mp3 파일로 저장해야 합니다.
```
3. 참고 자료
- OpenAI API 공식 문서 : https://developers.openai.com/api/docs
- 과금 확인 : https://platform.openai.com/settings/organization/billing
```


# a. 환경변수 사용 : API 키는 코드에 직접 포함하지 않고, .env 파일을 이용해 관리해야 합니다.

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

# b. 텍스트 파일 읽기 : 프로그램은 지정된 텍스트 파일(ch06_quiz.txt)을 읽어와야 합니다.

In [ ]:
with open('data/ch06_quiz.txt', 'r', encoding='utf-8') as txtf:
    txtfile = txtf.read()
print(txtfile)

# c. 텍스트 요약 : OpenAI API를 사용하여 긴 텍스트를 100자 이내로 요약해야 합니다.

In [ ]:
load_dotenv()
client = OpenAI()
response_summary = client.chat.completions.create(
    model = 'gpt-4.1-nano',
    messages = [
        {'role':'system','content':'You are an assistant that summarizes text.'},
        {'role':'user','content':f'Please summarize the provided sentences in Korean. Keep the summary under 100 characters. text : {txtfile}'}
    ] 
)

In [ ]:
summary = response_summary.choices[0].message.content
summary

# d. 음성 변환 : 요약된 내용을 OpenAI의 TTS(Text-to-Speech) API를 활용하여 음성데이터로 변환합니다.
# e. 지정된 보이스 사용 : OpenAI TTS에서 제공하는 특정 보이스("nova")를 사용해야 합니다.

In [ ]:
response_ko = client.audio.speech.create(
    model = 'tts-1-hd',
    voice = 'nova',
    input = summary,
    speed = 1.0
)

audio_data = response_ko.content

# f. MP3 파일 저장 : 생성된 음성 데이터를 ch6_quiz.mp3 파일로 저장해야 합니다.

In [ ]:
with open('data/ch6_quiz.mp3', 'wb') as audio_file:
    audio_file.write(audio_data)

In [ ]:
Audio('data/ch6_quiz.mp3', autoplay=True)